In [1]:
# confusion_matrix.ipynb - plot confusion matrix.

In [2]:
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import seaborn as sns
from logging import info, error
from matplotlib.patches import Patch
from sklearn.metrics import confusion_matrix

In [3]:
in_dir = '/groups/cgsd/xianjie/projects/cna-benchmark/HCC3N_600spot/tumor_identification/subclone/1r1n2t/base'
out_dir = '/groups/cgsd/xianjie/projects/cna-benchmark/HCC3N_600spot/tumor_identification/subclone/analysis/confusion_matrix_subclone'
cell_anno_fn = '/groups/cgsd/xianjie/projects/cna-benchmark/HCC3N_600spot/tumor_identification/subclone/1r1n2t/base/gen_data/simu/4_rs/rs.cell_anno.tsv'

In [11]:
def assert_e(path):
    """Assert file or folder exists, mimicking shell "test -e"."""
    assert path is not None and os.path.exists(path)

def plot_labels_confusion_matrix(
    tool_list,
    tool_fn_list,
    truth_fn,
    out_fig_fn,
    fig_width = None,
    fig_height = None,
    fig_nrow = None,
    fig_ncol = 3,
    fig_dpi = 300,
    verbose = True
):
    # check args.
    assert len(tool_list) == len(tool_fn_list)
    new_tool_list, new_tool_fn_list = [], []
    for tool, fn in zip(tool_list, tool_fn_list):
        if not os.path.exists(fn):
            print("W: '%s' file does not exist!" % tool)
        else:
            new_tool_list.append(tool)
            new_tool_fn_list.append(fn)
    tool_list, tool_fn_list = new_tool_list, new_tool_fn_list
    assert_e(truth_fn)


    # plot confusion matrix for each tool.
    if verbose:
        info("plot confusion matrix for each tool's labels ...")

    n = len(tool_list)
    fig_ncol = min(n, fig_ncol)
    if fig_nrow is None:
        fig_nrow = int(np.ceil(n / fig_ncol))
    if fig_width is None:
        fig_width = 2.5 * fig_ncol
    if fig_height is None:
        fig_height = 2.5 * fig_nrow
    
    # Create subplots with exact number needed (no empty subplots)
    fig, axes = plt.subplots(
        fig_nrow, fig_ncol,
        figsize = (fig_width, fig_height)
    )
    
    # Handle axes array - flatten if 2D, convert to list if single
    if n == 1:
        axes = [axes]
    elif fig_nrow == 1:
        axes = [axes] if not isinstance(axes, np.ndarray) else axes.tolist()
    else:
        axes = axes.flatten()
    
    # Hide unused subplots
    for idx in range(n, len(axes)):
        axes[idx].set_visible(False)

    # Publication-ready styling
    title_fontsize = 6
    label_fontsize = 6
    tick_fontsize = 5
    annot_fontsize = 6
    
    df = pd.read_csv(truth_fn, sep = '\t')
    truth_labels = df['annotation'].to_numpy()
    i = 0
    for ax, tool, tool_fn in zip(axes[:n], tool_list, tool_fn_list):
        display_name = tool
        if verbose:
            info("process %s ..." % display_name)

        df = pd.read_csv(tool_fn, sep = '\t')
        tool_labels = df['prediction'].to_numpy()
        print(len(truth_labels), len(tool_labels))
        cm = pd.crosstab(truth_labels, tool_labels)
        print(cm)
        if i == 0:
            sns.heatmap(
                cm, annot = True, fmt = 'd', cmap = 'Blues',
                xticklabels = ['Normal', 'Tumor'],
                yticklabels = ['Normal', 'Tumor1', 'Tumor2'],
                cbar = False,
                ax = ax,
                annot_kws = {'fontsize': annot_fontsize}
            )
        else:
            sns.heatmap(
                cm, annot = True, fmt = 'd', cmap = 'Blues',
                xticklabels = ['Normal', 'Tumor'],
                yticklabels = False,
                cbar = False,
                ax = ax,
                annot_kws = {'fontsize': annot_fontsize}
            )            
        #ax.set_title(f'{display_name}', fontsize = title_fontsize, pad = 8)
        ax.set_title(f'{display_name}', fontsize = title_fontsize)
        ax.set_xlabel('Predicted', fontsize = label_fontsize)
        if i == 0:
            ax.set_ylabel('True', fontsize = label_fontsize)
        else:
            ax.set_ylabel(None)
        ax.tick_params(labelsize = label_fontsize)
        
        ax.tick_params(
            axis = 'x',          # Target x-axis ticks
            labelsize = tick_fontsize,
            rotation = 0         # Force horizontal (critical fix)
        )
        if i == 0:
            ax.tick_params(
                axis = 'y',          # Keep y-axis ticks as-is (vertical by default)
                labelsize = tick_fontsize,
                rotation = 90
            )
        i += 1

    #plt.subplots_adjust(wspace = 0.1)
    plt.tight_layout()
    fig.savefig(out_fig_fn, dpi = fig_dpi, bbox_inches = 'tight')
    plt.close(fig)


    res = dict(
        out_fig_fn = out_fig_fn
    )
    return(res)

In [12]:
anno = pd.read_csv(cell_anno_fn, sep = '\t', header = None)
anno.columns = ['cell', 'clone']
anno

,cell,clone
0,AAACCTGAGCACACAG-1,ref
1,AAACCTGGTACCAGTT-1,ref
2,AAACCTGGTACCATCA-1,ref
3,AAACGGGAGCGCCTCA-1,ref
4,AAACGGGGTCGCTTTC-1,ref
...,...,...
2095,TTTGTCAAGACCTTTG-1,tumor2
2096,TTTGTCAGTATAAACG-1,tumor2
2097,TTTGTCAGTGTCAATC-1,tumor2
2098,TTTGTCAGTTGATTGC-1,tumor2


In [13]:
anno['clone'].value_counts()

clone
ref       600
normal    500
tumor1    500
tumor2    500
Name: count, dtype: int64

In [14]:
truth_fn = os.path.join(in_dir, 'tumor_identification/1_overlap/overlap.truth.tsv')
truth = pd.read_csv(truth_fn, sep = '\t')
truth.columns = ['cell', 'clone']
truth

,cell,clone
0,CAGTAACTCGAATCCA-1,normal
1,CAGTCCTAGGCTCAGA-1,normal
2,CAGTCCTAGGTGTGGT-1,normal
3,CAGTCCTCAGACAAAT-1,normal
4,CAGTCCTTCGGAGGTA-1,normal
...,...,...
1492,TTTGTCAAGACCTTTG-1,tumor
1493,TTTGTCAGTATAAACG-1,tumor
1494,TTTGTCAGTGTCAATC-1,tumor
1495,TTTGTCAGTTGATTGC-1,tumor


In [15]:
truth = anno.loc[anno['cell'].isin(truth['cell'])]
truth.columns = ['barcode', 'annotation']
truth

,barcode,annotation
600,CAGTAACTCGAATCCA-1,normal
601,CAGTCCTAGGCTCAGA-1,normal
602,CAGTCCTAGGTGTGGT-1,normal
603,CAGTCCTCAGACAAAT-1,normal
604,CAGTCCTTCGGAGGTA-1,normal
...,...,...
2095,TTTGTCAAGACCTTTG-1,tumor2
2096,TTTGTCAGTATAAACG-1,tumor2
2097,TTTGTCAGTGTCAATC-1,tumor2
2098,TTTGTCAGTTGATTGC-1,tumor2


In [16]:
truth.to_csv(
    os.path.join(out_dir, 'truth_subclone.tsv'),
    sep = '\t',
    header = True,
    index = False
)

In [17]:
tool_list = ['InferCNV', 'CopyKAT', 'Numbat', 'XClone', 'CalicoST']

plot_labels_confusion_matrix(
    tool_list = tool_list,
    tool_fn_list = [os.path.join(in_dir, \
        'tumor_identification/1_overlap/overlap.%s.tsv' % x.lower())   \
        for x in tool_list],
    truth_fn = os.path.join(out_dir, 'truth_subclone.tsv'),
    out_fig_fn = os.path.join(out_dir, 'subclone_confusion_matrix.png'),
    fig_width = 4.5,
    fig_height = 1.7,
    fig_nrow = 1,
    fig_ncol = 5,
    fig_dpi = 300,
    verbose = True
)

1497 1497
col_0   normal  tumor
row_0                
normal     498      0
tumor1     500      0
tumor2       0    499
1497 1497
col_0   normal  tumor
row_0                
normal     498      0
tumor1     500      0
tumor2       0    499
1497 1497
col_0   normal  tumor
row_0                
normal     495      3
tumor1       3    497
tumor2       0    499
1497 1497
col_0   normal  tumor
row_0                
normal     491      7
tumor1     495      5
tumor2      17    482
1497 1497
col_0   normal  tumor
row_0                
normal     462     36
tumor1     436     64
tumor2       0    499


{'out_fig_fn': '/groups/cgsd/xianjie/projects/cna-benchmark/HCC3N_600spot/tumor_identification/subclone/analysis/confusion_matrix_subclone/subclone_confusion_matrix.png'}